# Contador de dedos — Notebook 2: detector de mano (Faster R-CNN)

Fine-tuning de **Faster R-CNN** (la misma familia que en P3) para detectar **manos** en imágenes de webcam.
La caja que se obtiene se recorta, se preprocesa y se pasa al clasificador del Notebook 1.

| Paso | Tema de la asignatura |
|---|---|
| Faster R-CNN: backbone + FPN + RPN + anchors + cabeza | DL4, P3 |
| Transfer learning: se cambia la cabeza a 2 clases (fondo, mano) | DL1, DL4 |
| Train/val/test **separados por persona** (sin fuga de datos) | DL1 |
| SGD + momentum, *warmup* de la tasa de aprendizaje | DL2 |
| IoU, NMS, curva P-R, AP, mAP@0.5 y mAP COCO (0.5:0.95) | DL4 |

**Dataset:** HaGRID (*HAnd Gesture Recognition Image Dataset*), muestra de 30k imágenes a 384p.
Son personas **delante de una cámara**, como tu webcam, así que es mucho más parecido a tu caso que EgoHands (vídeo en primera persona).
Usamos **todas** las cajas de mano (incluidas las `no_gesture`) como una única clase: *mano*.

> Entorno: `GPU (T4)`. Con la configuración por defecto (6.000 imágenes, 6 épocas) tarda unos 20–30 min.

## 0. Preparación

In [ ]:
!pip -q install torchmetrics pycocotools

In [ ]:
%%writefile preprocesado.py
"""
preprocesado.py — Preprocesado clásico COMPARTIDO entre entrenamiento (Colab) y uso real (webcam).

Idea clave (P3: "el preprocesamiento es parte del modelo"):
las imágenes de Kaggle y los recortes de la webcam deben pasar por la MISMA cadena
para que la red vea siempre el mismo tipo de imagen: una silueta binaria de la mano,
centrada, cuadrada y de 64x64.

Cadena (temas Clase2/Clase3 y P1):
  1. Suavizado gaussiano (eliminación de ruido)
  2. Segmentación:
       - Kaggle (gris, mano clara sobre fondo oscuro): umbral de Otsu
       - Webcam (color): piel en YCrCb  AND  Otsu sobre el canal Cr
  3. Morfología: opening (quita ruido) + closing (tapa huecos)
  4. Etiquetado de blobs: nos quedamos con la componente conexa más grande
  5. Recorte al blob + padding para hacerlo cuadrado (no deformar los dedos)
  6. Redimensionado a 64x64
"""
import cv2
import numpy as np

TAM_SALIDA = 64

# Rango de piel típico en YCrCb (Chai & Ngan). Ajustable desde la app con las teclas.
CR_MIN, CR_MAX = 133, 173
CB_MIN, CB_MAX = 77, 127


def _limpiar_mascara(mascara, k_open=3, k_close=7):
    """Opening + closing con elemento estructurante elíptico."""
    ee_o = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_open, k_open))
    ee_c = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k_close, k_close))
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_OPEN, ee_o)
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_CLOSE, ee_c)
    return mascara


def _mayor_blob(mascara):
    """Etiquetado (blob labelling) y selección de la componente conexa de mayor área."""
    n, etiquetas, stats, _ = cv2.connectedComponentsWithStats(mascara, connectivity=8)
    if n <= 1:  # solo fondo
        return None, None
    idx = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    blob = np.where(etiquetas == idx, 255, 0).astype(np.uint8)
    x, y, w, h = stats[idx, :4]
    return blob, (x, y, w, h)


def _cuadrar_y_redimensionar(blob, caja, tam=TAM_SALIDA, margen=0.10):
    """Recorta al blob, añade un margen, rellena a cuadrado y redimensiona."""
    x, y, w, h = caja
    recorte = blob[y:y + h, x:x + w]
    lado = int(max(w, h) * (1 + 2 * margen))
    lienzo = np.zeros((lado, lado), np.uint8)
    oy, ox = (lado - h) // 2, (lado - w) // 2
    lienzo[oy:oy + h, ox:ox + w] = recorte
    # INTER_AREA para reducir; luego se vuelve a binarizar para quitar grises
    peq = cv2.resize(lienzo, (tam, tam), interpolation=cv2.INTER_AREA)
    _, peq = cv2.threshold(peq, 127, 255, cv2.THRESH_BINARY)
    return peq


def mascara_kaggle(gris):
    """Segmentación para el dataset de Kaggle: mano clara sobre fondo oscuro -> Otsu."""
    suave = cv2.GaussianBlur(gris, (5, 5), 0)
    _, mascara = cv2.threshold(suave, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return mascara


def mascara_webcam(bgr, cr_min=CR_MIN, cr_max=CR_MAX, cb_min=CB_MIN, cb_max=CB_MAX,
                   usar_clahe=True):
    """Segmentación de piel para recortes de webcam: YCrCb + Otsu sobre Cr."""
    ycrcb = cv2.cvtColor(bgr, cv2.COLOR_BGR2YCrCb)
    if usar_clahe:  # CLAHE sobre la luminancia (P1): robustez a la iluminación
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        ycrcb[:, :, 0] = clahe.apply(ycrcb[:, :, 0])
    ycrcb = cv2.GaussianBlur(ycrcb, (5, 5), 0)
    piel = cv2.inRange(ycrcb, (0, cr_min, cb_min), (255, cr_max, cb_max))
    _, otsu_cr = cv2.threshold(ycrcb[:, :, 1], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return cv2.bitwise_and(piel, otsu_cr)


def preprocesar_mano(imagen, origen="kaggle", tam=TAM_SALIDA, devolver_mascara=False, **kw):
    """
    Devuelve una silueta binaria (uint8, 0/255) de tam x tam, o None si no hay mano.
      origen="kaggle": imagen en gris (o BGR) del dataset de Kaggle
      origen="webcam": recorte BGR de la bounding box del detector
    """
    if origen == "kaggle":
        gris = imagen if imagen.ndim == 2 else cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)
        mascara = mascara_kaggle(gris)
    elif origen == "webcam":
        mascara = mascara_webcam(imagen, **kw)
    else:
        raise ValueError("origen debe ser 'kaggle' o 'webcam'")

    mascara = _limpiar_mascara(mascara)
    blob, caja = _mayor_blob(mascara)
    if blob is None or caja[2] * caja[3] < 0.01 * mascara.size:
        return (None, mascara) if devolver_mascara else None
    salida = _cuadrar_y_redimensionar(blob, caja, tam)
    return (salida, mascara) if devolver_mascara else salida


def contar_dedos_clasico(silueta):
    """
    LÍNEA BASE SIN DEEP LEARNING (Clase3: descripción de objetos).
    Cuenta dedos con la envolvente convexa y los defectos de convexidad.
    Útil para comparar 'clásico vs. deep learning' en la memoria.
    """
    contornos, _ = cv2.findContours(silueta, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contornos:
        return 0
    c = max(contornos, key=cv2.contourArea)
    hull = cv2.convexHull(c, returnPoints=False)
    if hull is None or len(hull) < 4:
        return 0
    try:
        defectos = cv2.convexityDefects(c, hull)
    except cv2.error:
        return 0
    if defectos is None:
        return 0
    _, _, _, h = cv2.boundingRect(c)
    valles = 0
    for s, e, f, d in defectos.reshape(-1, 4):
        ini, fin, prof = c[s][0], c[e][0], c[f][0]
        a = np.linalg.norm(fin - ini)
        b = np.linalg.norm(prof - ini)
        cc = np.linalg.norm(fin - prof)
        if b * cc == 0:
            continue
        angulo = np.degrees(np.arccos(np.clip((b**2 + cc**2 - a**2) / (2 * b * cc), -1, 1)))
        # Valle entre dedos: ángulo agudo y suficientemente profundo
        if angulo < 90 and d / 256.0 > 0.10 * h:
            valles += 1
    if valles == 0:
        # 0 o 1 dedo: se distingue por la elongación de la silueta
        x, y, w, hh = cv2.boundingRect(c)
        area_ratio = cv2.contourArea(c) / float(w * hh)
        return 1 if area_ratio < 0.55 else 0
    return min(valles + 1, 5)

In [ ]:
%%writefile modelos.py
"""
modelos.py — Definición de las redes, COMPARTIDA entre Colab y la app local.

Clasificadores (6 clases: 0..5 dedos), entrada 3x64x64 (silueta replicada en 3 canales):
  - simplecnn         : CNN pequeña entrenada desde cero (línea base, como en P2)
  - resnet18_frozen   : ResNet18 ImageNet, solo se entrena la cabeza (transfer learning)
  - resnet18_finetune : ResNet18 ImageNet, se reentrenan layer3, layer4 y la cabeza (fine-tuning)
  - mobilenetv3       : MobileNetV3-Small ImageNet con fine-tuning (ligera, convs depthwise)

Detector de mano:
  - Faster R-CNN con backbone MobileNetV3-Large + FPN (misma familia que el de P3)
"""
import torch
import torch.nn as nn
from torchvision import models

NUM_CLASES = 6
MEDIA_IMAGENET = [0.485, 0.456, 0.406]
STD_IMAGENET = [0.229, 0.224, 0.225]


class SimpleCNN(nn.Module):
    """CNN desde cero inspirada en la de P2: (Conv-BN-ReLU)x2 + MaxPool, tres bloques."""

    def __init__(self, num_clases=NUM_CLASES, p_dropout=0.5):
        super().__init__()

        def bloque(cin, cout):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        self.features = nn.Sequential(bloque(3, 32), bloque(32, 64), bloque(64, 128))
        self.pool = nn.AdaptiveAvgPool2d(1)  # global average pooling: vale para 64 o 112 px
        self.clasificador = nn.Sequential(
            nn.Flatten(), nn.Dropout(p_dropout), nn.Linear(128, 128),
            nn.ReLU(inplace=True), nn.Dropout(p_dropout), nn.Linear(128, num_clases),
        )

    def forward(self, x):
        return self.clasificador(self.pool(self.features(x)))


def crear_clasificador(nombre, preentrenado=True, num_clases=NUM_CLASES):
    """Devuelve el modelo indicado con la cabeza adaptada a num_clases."""
    if nombre == "simplecnn":
        return SimpleCNN(num_clases)

    if nombre.startswith("resnet18"):
        pesos = models.ResNet18_Weights.DEFAULT if preentrenado else None
        m = models.resnet18(weights=pesos)
        m.fc = nn.Linear(m.fc.in_features, num_clases)
        for p in m.parameters():
            p.requires_grad = False
        entrenables = [m.fc]
        if nombre == "resnet18_finetune":
            entrenables += [m.layer3, m.layer4]
        for mod in entrenables:
            for p in mod.parameters():
                p.requires_grad = True
        return m

    if nombre == "mobilenetv3":
        pesos = models.MobileNet_V3_Small_Weights.DEFAULT if preentrenado else None
        m = models.mobilenet_v3_small(weights=pesos)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_clases)
        return m  # fine-tuning completo (es pequeña)

    raise ValueError(f"Modelo desconocido: {nombre}")


def capa_gradcam(modelo, nombre):
    """Última capa convolucional, para Grad-CAM."""
    if nombre == "simplecnn":
        return modelo.features[-1]
    if nombre.startswith("resnet18"):
        return modelo.layer4
    if nombre == "mobilenetv3":
        return modelo.features[-1]
    raise ValueError(nombre)


def crear_detector(num_clases=2, preentrenado=True, min_size=480, max_size=640):
    """
    Faster R-CNN MobileNetV3-Large FPN (torchvision) con la cabeza cambiada a
    num_clases (fondo + mano). Misma API que el fasterrcnn_resnet50_fpn_v2 de P3.
    """
    from torchvision.models.detection import fasterrcnn_mobilenet_v3_large_fpn
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

    pesos = "DEFAULT" if preentrenado else None
    m = fasterrcnn_mobilenet_v3_large_fpn(weights=pesos, weights_backbone=None,
                                          min_size=min_size, max_size=max_size)
    in_feat = m.roi_heads.box_predictor.cls_score.in_features
    m.roi_heads.box_predictor = FastRCNNPredictor(in_feat, num_clases)
    return m


def silueta_a_tensor(silueta, tam=64):
    """uint8 (H,W) 0/255 -> tensor (1,3,tam,tam) normalizado como ImageNet."""
    import cv2
    if silueta.shape[0] != tam:
        silueta = cv2.resize(silueta, (tam, tam), interpolation=cv2.INTER_NEAREST)
    t = torch.from_numpy(silueta).float().div(255.0)
    t = t.unsqueeze(0).repeat(3, 1, 1)
    media = torch.tensor(MEDIA_IMAGENET).view(3, 1, 1)
    std = torch.tensor(STD_IMAGENET).view(3, 1, 1)
    return ((t - media) / std).unsqueeze(0)

In [ ]:
import os, glob, json, time, copy, random, math
from pathlib import Path
import numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt, matplotlib.patches as patches
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import tv_tensors
from torchvision.transforms import v2 as T
from torchvision.ops import box_iou
from torchmetrics.detection import MeanAveragePrecision
from sklearn.model_selection import GroupShuffleSplit

import importlib, preprocesado, modelos
importlib.reload(preprocesado); importlib.reload(modelos)
from modelos import crear_detector
from preprocesado import preprocesar_mano

SEMILLA = 42
random.seed(SEMILLA); np.random.seed(SEMILLA); torch.manual_seed(SEMILLA)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

## 1. Descarga de HaGRID-30k-384p (≈1 GB) desde Hugging Face

In [ ]:
from huggingface_hub import hf_hub_download
zip_path = hf_hub_download(repo_id="cj-mills/hagrid-sample-30k-384p",
                           filename="hagrid-sample-30k-384p.zip", repo_type="dataset")
RAIZ = Path("/content/hagrid")
if not (RAIZ / "hagrid-sample-30k-384p").exists():
    !unzip -q "{zip_path}" -d /content/hagrid
DIR = RAIZ / "hagrid-sample-30k-384p"
print(os.listdir(DIR))

## 2. Anotaciones
Cada `.json` de `ann_train_val/` contiene, por imagen: `bboxes` en formato **[x, y, ancho, alto] normalizado (0–1)**, `labels` (gesto o `no_gesture`) y `user_id`.
Los json traen anotaciones de muchas más imágenes de las incluidas en esta muestra, así que nos quedamos solo con las que existen.

In [ ]:
imagenes = {Path(p).stem: p for p in glob.glob(str(DIR / "hagrid_30k" / "*" / "*.jpg"))}
filas = []
for jf in sorted(glob.glob(str(DIR / "ann_train_val" / "*.json"))):
    for img_id, a in json.load(open(jf)).items():
        if img_id in imagenes and len(a["bboxes"]):
            filas.append({"id": img_id, "ruta": imagenes[img_id], "bboxes": a["bboxes"],
                          "labels": a["labels"], "user_id": a["user_id"], "gesto": Path(jf).stem})
ann = pd.DataFrame(filas)
ann["n_manos"] = ann.bboxes.apply(len)
print(len(imagenes), "imágenes |", len(ann), "anotadas |", ann.user_id.nunique(), "personas")
print(ann.n_manos.value_counts().sort_index().rename("imágenes con N manos"))

## 3. División train / val / test **por persona** (DL1)
Si la misma persona apareciera en train y en test, el test sería demasiado optimista (*fuga de datos*).
Por eso separamos por `user_id`. Para que el entrenamiento sea rápido, tomamos un subconjunto (puedes subir `N_TRAIN`).

In [ ]:
N_TRAIN, N_VAL, N_TEST = 6000, 800, 800

gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEMILLA)
i_tr, i_resto = next(gss.split(ann, groups=ann.user_id))
resto = ann.iloc[i_resto]
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=SEMILLA)
i_va, i_te = next(gss2.split(resto, groups=resto.user_id))

df_tr = ann.iloc[i_tr].sample(min(N_TRAIN, len(i_tr)), random_state=SEMILLA)
df_va = resto.iloc[i_va].sample(min(N_VAL, len(i_va)), random_state=SEMILLA)
df_te = resto.iloc[i_te].sample(min(N_TEST, len(i_te)), random_state=SEMILLA)
assert not (set(df_tr.user_id) & set(df_te.user_id)), "¡Hay personas repetidas entre train y test!"
print("train", len(df_tr), "| val", len(df_va), "| test", len(df_te))

## 4. Dataset y *data augmentation* para detección
En detección, las transformaciones geométricas **deben mover también las cajas**. `torchvision.transforms.v2` lo hace solo si las cajas se envuelven en `tv_tensors.BoundingBoxes`.
- Geométricas: volteo horizontal y pequeño zoom/recorte.
- Fotométricas: brillo, contraste, saturación y tono (variaciones de iluminación de la webcam).

In [ ]:
class ManosDataset(Dataset):
    def __init__(self, df, entrenamiento=False):
        self.df = df.reset_index(drop=True)
        if entrenamiento:
            self.tf = T.Compose([
                T.RandomHorizontalFlip(0.5),
                T.RandomApply([T.RandomZoomOut(fill=114, side_range=(1.0, 1.5))], p=0.3),
                T.ColorJitter(0.3, 0.3, 0.3, 0.05),
                T.SanitizeBoundingBoxes(),
            ])
        else:
            self.tf = T.Identity()

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        f = self.df.iloc[i]
        img = cv2.cvtColor(cv2.imread(f.ruta), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        cajas = np.array(f.bboxes, dtype=np.float32).reshape(-1, 4)
        xyxy = np.stack([cajas[:, 0] * w, cajas[:, 1] * h,
                         (cajas[:, 0] + cajas[:, 2]) * w, (cajas[:, 1] + cajas[:, 3]) * h], 1)
        img = tv_tensors.Image(torch.from_numpy(img).permute(2, 0, 1))
        target = {"boxes": tv_tensors.BoundingBoxes(torch.from_numpy(xyxy), format="XYXY", canvas_size=(h, w)),
                  "labels": torch.ones(len(xyxy), dtype=torch.int64)}      # 1 = mano
        img, target = self.tf(img, target)
        return img.float() / 255.0, {"boxes": target["boxes"].as_subclass(torch.Tensor), "labels": target["labels"]}

collate = lambda b: tuple(zip(*b))
BATCH = 8
dl_tr = DataLoader(ManosDataset(df_tr, True), batch_size=BATCH, shuffle=True, num_workers=2, collate_fn=collate)
dl_va = DataLoader(ManosDataset(df_va), batch_size=BATCH, num_workers=2, collate_fn=collate)
dl_te = DataLoader(ManosDataset(df_te), batch_size=BATCH, num_workers=2, collate_fn=collate)

def dibujar(ax, img, cajas, color="lime", scores=None, titulo=""):
    ax.imshow(img.permute(1, 2, 0).numpy() if torch.is_tensor(img) else img)
    for k, b in enumerate(cajas):
        x1, y1, x2, y2 = [float(v) for v in b]
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color=color, lw=2))
        if scores is not None: ax.text(x1, y1 - 3, f"{float(scores[k]):.2f}", color=color, fontsize=8)
    ax.set_title(titulo, fontsize=9); ax.axis("off")

imgs, tgts = next(iter(dl_tr))
fig, ax = plt.subplots(1, 4, figsize=(16, 4))
for a, im, t in zip(ax, imgs, tgts): dibujar(a, im, t["boxes"], titulo=f"{len(t['boxes'])} mano(s)")
plt.suptitle("Ground truth (con aumento)");

## 5. Modelo: Faster R-CNN MobileNetV3-Large FPN
Es la misma API que el `fasterrcnn_resnet50_fpn_v2` de P3, pero con un backbone ligero, para que vaya bien en tiempo real en tu PC.
Cargamos los pesos de **COCO** (que ya incluye la clase *persona*) y **sustituimos la cabeza** (`FastRCNNPredictor`) por una de 2 clases: fondo y mano.

Las 4 pérdidas que se ven al entrenar corresponden a las 2 etapas del detector (DL4):
- **RPN**: `loss_objectness` (¿hay objeto en el anchor?) y `loss_rpn_box_reg` (ajuste del anchor).
- **Cabeza**: `loss_classifier` (¿mano o fondo?) y `loss_box_reg` (ajuste final de la caja).

In [ ]:
detector = crear_detector(num_clases=2, preentrenado=True, min_size=384, max_size=640).to(DEVICE)
n_tot = sum(p.numel() for p in detector.parameters()); n_ent = sum(p.numel() for p in detector.parameters() if p.requires_grad)
print(f"Parámetros: {n_tot/1e6:.1f} M (entrenables {n_ent/1e6:.1f} M)")
print(detector.rpn.anchor_generator.sizes, detector.rpn.anchor_generator.aspect_ratios[0])

## 6. Evaluación: AP implementada a mano + mAP de `torchmetrics`
Implementamos la **AP con interpolación en todos los puntos** (DL4) para entender qué es; luego la comparamos con `torchmetrics`, que da también la **mAP COCO** (promedio sobre IoU 0.5:0.95).

In [ ]:
@torch.no_grad()
def predecir(modelo, dl, max_lotes=None):
    modelo.eval(); preds, gts = [], []
    for k, (imgs, tgts) in enumerate(dl):
        if max_lotes and k >= max_lotes: break
        out = modelo([im.to(DEVICE) for im in imgs])
        preds += [{kk: v.cpu() for kk, v in o.items()} for o in out]; gts += list(tgts)
    return preds, gts

def curva_pr(preds, gts, umbral_iou=0.5):
    # Ordena todas las detecciones por confianza y marca TP/FP (DL4)
    dets = []
    for i, p in enumerate(preds):
        for b, s in zip(p["boxes"], p["scores"]): dets.append((float(s), i, b))
    dets.sort(key=lambda d: -d[0])
    usados = [torch.zeros(len(g["boxes"]), dtype=torch.bool) for g in gts]
    n_gt = sum(len(g["boxes"]) for g in gts); tp = np.zeros(len(dets)); fp = np.zeros(len(dets))
    for k, (s, i, b) in enumerate(dets):
        g = gts[i]["boxes"]
        if len(g):
            ious = box_iou(b[None], g)[0]; j = int(ious.argmax())
            if ious[j] >= umbral_iou and not usados[i][j]:
                tp[k] = 1; usados[i][j] = True; continue
        fp[k] = 1
    tp_c, fp_c = np.cumsum(tp), np.cumsum(fp)
    rec = tp_c / max(n_gt, 1); prec = tp_c / np.maximum(tp_c + fp_c, 1e-9)
    # AP con interpolación en todos los puntos: envolvente monótona de la precisión
    mrec = np.concatenate([[0], rec, [1]]); mpre = np.concatenate([[1], prec, [0]])
    for k in range(len(mpre) - 2, -1, -1): mpre[k] = max(mpre[k], mpre[k + 1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    ap = float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))
    return prec, rec, np.array([d[0] for d in dets]), ap

def map_torchmetrics(preds, gts):
    m = MeanAveragePrecision(iou_type="bbox"); m.update(preds, gts); r = m.compute()
    return {k: float(r[k]) for k in ["map", "map_50", "map_75", "map_small", "map_medium", "map_large", "mar_100"]}

## 7. Entrenamiento
- **SGD + momentum 0.9** y *weight decay* (la receta habitual de Faster R-CNN).
- ***Warmup* lineal** durante las primeras iteraciones (DL2): una tasa alta desde el principio puede disparar la pérdida.
- Después, decaimiento **coseno**. Nos quedamos con los pesos de mejor **mAP@0.5 en validación**.

In [ ]:
EPOCAS, LR, WARMUP = 6, 0.01, 300
params = [p for p in detector.parameters() if p.requires_grad]
opt = torch.optim.SGD(params, lr=LR, momentum=0.9, weight_decay=1e-4)
total_it = EPOCAS * len(dl_tr)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda it: (it + 1) / WARMUP if it < WARMUP
                                          else 0.5 * (1 + math.cos(math.pi * (it - WARMUP) / max(1, total_it - WARMUP))))
scaler = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")

hist, mejor_map, mejor_estado = [], -1, None
for ep in range(EPOCAS):
    detector.train(); t0 = time.time(); acum = {}
    for it, (imgs, tgts) in enumerate(dl_tr):
        imgs = [im.to(DEVICE) for im in imgs]
        tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in tgts]
        with torch.autocast(DEVICE.type, enabled=DEVICE.type == "cuda"):
            perdidas = detector(imgs, tgts)
            loss = sum(perdidas.values())
        opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        for k, v in perdidas.items(): acum[k] = acum.get(k, 0) + v.item()
        if it % 100 == 0:
            print(f"  ép {ep+1} it {it}/{len(dl_tr)} loss {loss.item():.3f} lr {opt.param_groups[0]['lr']:.4f}")
    p, g = predecir(detector, dl_va)
    m = map_torchmetrics(p, g)
    fila = {"epoca": ep + 1, **{k: v / len(dl_tr) for k, v in acum.items()}, **m}; hist.append(fila)
    print(f"ÉPOCA {ep+1} | {time.time()-t0:.0f}s | val mAP@0.5 {m['map_50']:.3f} | mAP 0.5:0.95 {m['map']:.3f}")
    if m["map_50"] > mejor_map:
        mejor_map, mejor_estado = m["map_50"], copy.deepcopy(detector.state_dict())
detector.load_state_dict(mejor_estado)
hist = pd.DataFrame(hist).set_index("epoca"); hist.round(4)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
hist[["loss_classifier", "loss_box_reg", "loss_objectness", "loss_rpn_box_reg"]].plot(ax=ax[0], title="Pérdidas de entrenamiento")
hist[["map_50", "map", "mar_100"]].plot(ax=ax[1], title="Validación", marker="o");

## 8. Resultados en test: curva P-R, AP y mAP

In [ ]:
p_te, g_te = predecir(detector, dl_te)
prec, rec, conf, ap50 = curva_pr(p_te, g_te, 0.5)
_, _, _, ap75 = curva_pr(p_te, g_te, 0.75)
m_te = map_torchmetrics(p_te, g_te)
print(f"AP@0.5 (implementación propia): {ap50:.3f} | torchmetrics: {m_te['map_50']:.3f}  (COCO interpola en 101 puntos → pequeñas diferencias)")
print(f"AP@0.75 propia: {ap75:.3f} | mAP COCO 0.5:0.95: {m_te['map']:.3f}")
print({k: round(v, 3) for k, v in m_te.items()})

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(rec, prec); ax[0].set_xlabel("Recall"); ax[0].set_ylabel("Precision"); ax[0].set_title(f"Curva P-R (IoU≥0.5) · AP={ap50:.3f}")
ax[1].plot(conf, prec, label="precision"); ax[1].plot(conf, rec, label="recall"); ax[1].invert_xaxis()
ax[1].set_xlabel("umbral de confianza"); ax[1].legend(); ax[1].set_title("Elegir el umbral de confianza para la app");

## 9. Efecto del umbral de confianza y de la NMS (DL4)
- `box_score_thresh`: por debajo de este valor se descartan las detecciones.
- `box_nms_thresh`: dos cajas con IoU mayor que este valor se consideran la misma y se queda la de mayor confianza.
Con `nms_thresh = 1.0` (sin NMS) aparecen cajas duplicadas.

In [ ]:
imgs, tgts = next(iter(dl_te))
config = [(0.05, 1.0, "sin NMS, umbral 0.05"), (0.05, 0.5, "NMS 0.5, umbral 0.05"), (0.5, 0.5, "NMS 0.5, umbral 0.5")]
fig, ax = plt.subplots(len(config), 4, figsize=(16, 4 * len(config)))
for r, (sc, nms, tit) in enumerate(config):
    det = copy.deepcopy(detector).eval(); det.roi_heads.score_thresh = sc; det.roi_heads.nms_thresh = nms
    with torch.no_grad(): out = det([im.to(DEVICE) for im in imgs[:4]])
    for c in range(4):
        dibujar(ax[r, c], imgs[c], tgts[c]["boxes"], color="lime")
        dibujar(ax[r, c], imgs[c], out[c]["boxes"].cpu(), color="red", scores=out[c]["scores"].cpu(),
                titulo=f"{tit} · {len(out[c]['boxes'])} cajas")
plt.suptitle("verde = ground truth · rojo = predicción"); plt.tight_layout()

## 10. Velocidad en CPU (lo que tendrá tu PC)
Si el detector va lento en CPU, la app **solo detecta cada N fotogramas** y, entre medias, mueve la caja con un **filtro de Kalman** (DL7).

In [ ]:
det_cpu = copy.deepcopy(detector).cpu().eval(); x = torch.rand(3, 480, 640)
with torch.no_grad():
    for _ in range(3): det_cpu([x])
    t0 = time.time(); [det_cpu([x]) for _ in range(10)]
ms = (time.time() - t0) / 10 * 1000
print(f"Detector en CPU: {ms:.0f} ms/fotograma → {1000/ms:.1f} FPS. Detectando 1 de cada 3 fotogramas, el detector solo limita a ≈ {3000/ms:.1f} FPS de vídeo.")

## 11. Prueba con tu webcam: detector + recorte + silueta
Esta celda junta los dos notebooks: detecta tu mano, la recorta con margen y la pasa por `preprocesar_mano(..., "webcam")`.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def foto_webcam(nombre="foto.jpg"):
    display(Javascript('''
    async function takePhoto() {
      const div = document.createElement('div'); const btn = document.createElement('button'); btn.textContent = 'Capturar';
      div.appendChild(btn); const video = document.createElement('video'); video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      document.body.appendChild(div); div.appendChild(video); video.srcObject = stream; await video.play();
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
      await new Promise((r) => btn.onclick = r);
      const c = document.createElement('canvas'); c.width = video.videoWidth; c.height = video.videoHeight;
      c.getContext('2d').drawImage(video, 0, 0); stream.getVideoTracks()[0].stop(); div.remove();
      return c.toDataURL('image/jpeg', 0.9); }'''))
    open(nombre, "wb").write(b64decode(eval_js("takePhoto()").split(",")[1]))
    return cv2.imread(nombre)

def probar_foto(bgr, umbral=0.6, margen=0.15):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    with torch.no_grad():
        o = detector.eval()([torch.from_numpy(rgb).permute(2, 0, 1).float().div(255).to(DEVICE)])[0]
    keep = o["scores"] >= umbral
    cajas, scores = o["boxes"][keep].cpu(), o["scores"][keep].cpu()
    fig, ax = plt.subplots(1, 3, figsize=(14, 4)); dibujar(ax[0], rgb, cajas, "red", scores, f"{len(cajas)} mano(s)")
    if len(cajas):
        x1, y1, x2, y2 = cajas[0].tolist(); mw, mh = (x2 - x1) * margen, (y2 - y1) * margen
        H, W = bgr.shape[:2]
        rec = bgr[max(0, int(y1 - mh)):min(H, int(y2 + mh)), max(0, int(x1 - mw)):min(W, int(x2 + mw))]
        sil = preprocesar_mano(rec, "webcam")
        ax[1].imshow(cv2.cvtColor(rec, cv2.COLOR_BGR2RGB)); ax[1].set_title("recorte")
        if sil is not None: ax[2].imshow(sil, cmap="gray"); ax[2].set_title("silueta 64x64 → clasificador")
    for a in ax[1:]: a.axis("off")

probar_foto(foto_webcam())

## 12. Exportar el detector

In [ ]:
torch.save({"state_dict": detector.cpu().state_dict(), "min_size": 384, "max_size": 640, "num_clases": 2,
            "test": m_te}, "detector_mano.pth")
hist.to_csv("historial_detector.csv")
!zip -q detector_mano.zip detector_mano.pth historial_detector.csv
from google.colab import files; files.download("detector_mano.zip")

## Ideas para ampliar (y para la memoria)
- **Comparar backbones**: entrenar también `fasterrcnn_resnet50_fpn_v2` (el de P3) → tabla mAP vs. ms/fotograma (compromiso precisión–velocidad, DL3/DL4).
- **Mask R-CNN (P4)** en lugar de Faster R-CNN: obtendrías directamente la máscara de la mano y podrías comparar *segmentación con DL* frente a *piel YCrCb + Otsu*. Necesita un dataset con máscaras (por ejemplo, EgoHands trae polígonos).
- Añadir a train ~100 fotos tuyas anotadas (con CVAT o LabelMe, DL2) y medir la mejora en tu propio entorno.